In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
AZURE_OPENAI_EMBEDDING_DEPLOYMENT = os.getenv("AZURE_OPENAI_EMBEDDING_DEPLOYMENT")
AZURE_SEARCH_ENDPOINT = os.getenv("AZURE_SEARCH_ENDPOINT")
AZURE_SEARCH_API_KEY = os.getenv("AZURE_SEARCH_API_KEY")

print("✅ Environment variables loaded")
print("OpenAI Endpoint:", AZURE_OPENAI_ENDPOINT)
print("Search Endpoint:", AZURE_SEARCH_ENDPOINT)

✅ Environment variables loaded
OpenAI Endpoint: https://rag-chatbotrix-openai.openai.azure.com/
Search Endpoint: https://rag-chatbotrix-search.search.windows.net


In [3]:
def load_and_chunk(file_path, chunk_size=500, overlap=100):
    with open(file_path, "r", encoding="utf-8") as f:
        text = f.read()
    
    words = text.split()
    chunks = []
    start = 0
    
    while start < len(words):
        end = start + chunk_size
        chunk = " ".join(words[start:end])
        chunks.append(chunk)
        start += chunk_size - overlap
    
    return chunks

chunks = load_and_chunk("docs/sample.txt")

print(f"✅ Document loaded and chunked")
print(f"Total chunks: {len(chunks)}")
print(f"\nFirst chunk preview:\n{chunks[0]}")

✅ Document loaded and chunked
Total chunks: 1

First chunk preview:
Contoso Electronics offers a range of laptops and accessories. Our return policy allows returns within 30 days of purchase. All products come with a 1-year manufacturer warranty. For support, customers can contact us at support@contoso.com. Our store is open Monday to Friday, 9am to 6pm.


In [4]:
from openai import AzureOpenAI

openai_client = AzureOpenAI(
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
    api_version="2024-02-01"
)

def embed_chunks(chunks):
    embeddings = []
    for i, chunk in enumerate(chunks):
        response = openai_client.embeddings.create(
            input=chunk,
            model=AZURE_OPENAI_EMBEDDING_DEPLOYMENT
        )
        embeddings.append(response.data[0].embedding)
        print(f"✅ Embedded chunk {i+1}/{len(chunks)}")
    return embeddings

embeddings = embed_chunks(chunks)

print(f"\n✅ All chunks embedded")
print(f"Embedding vector length: {len(embeddings[0])}")

✅ Embedded chunk 1/1

✅ All chunks embedded
Embedding vector length: 1536


In [5]:
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    SearchIndex, SimpleField, SearchableField, SearchField,
    SearchFieldDataType, VectorSearch, HnswAlgorithmConfiguration,
    VectorSearchProfile
)
from azure.core.credentials import AzureKeyCredential

index_client = SearchIndexClient(
    endpoint=AZURE_SEARCH_ENDPOINT,
    credential=AzureKeyCredential(AZURE_SEARCH_API_KEY)
)

INDEX_NAME = "rag-chatbotrix-index"

fields = [
    SimpleField(name="id", type=SearchFieldDataType.String, key=True),
    SearchableField(name="content", type=SearchFieldDataType.String),
    SearchField(
        name="embedding",
        type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
        searchable=True,
        vector_search_dimensions=1536,
        vector_search_profile_name="my-vector-profile"
    )
]

vector_search = VectorSearch(
    algorithms=[HnswAlgorithmConfiguration(name="my-hnsw")],
    profiles=[VectorSearchProfile(name="my-vector-profile", algorithm_configuration_name="my-hnsw")]
)

index = SearchIndex(name=INDEX_NAME, fields=fields, vector_search=vector_search)
index_client.create_or_update_index(index)

print(f"✅ Index '{INDEX_NAME}' created successfully")

✅ Index 'rag-chatbotrix-index' created successfully


In [6]:
from azure.search.documents import SearchClient

search_client = SearchClient(
    endpoint=AZURE_SEARCH_ENDPOINT,
    index_name=INDEX_NAME,
    credential=AzureKeyCredential(AZURE_SEARCH_API_KEY)
)

documents = []
for i, (chunk, embedding) in enumerate(zip(chunks, embeddings)):
    documents.append({
        "id": str(i),
        "content": chunk,
        "embedding": embedding
    })

result = search_client.upload_documents(documents)

print(f"✅ Uploaded {len(documents)} document(s) to index")
print(f"Result: {result[0].succeeded}")

✅ Uploaded 1 document(s) to index
Result: True
